Smoothing Images 
================

Goals
-----

Learn to:
    -   Blur images with various low pass filters
    -   Apply custom-made filters to images (2D convolution)

2D Convolution ( Image Filtering )
----------------------------------

As in one-dimensional signals, images also can be filtered with various low-pass filters (LPF),
high-pass filters (HPF), etc. LPF helps in removing noise, blurring images, etc. HPF filters help
in finding edges in images.

OpenCV provides a function **cv.filter2D()** to convolve a kernel with an image. As an example, we
will try an averaging filter on an image. A 5x5 averaging filter kernel will look like the below:

$$K =  \frac{1}{25} \begin{bmatrix} 1 & 1 & 1 & 1 & 1  \\ 1 & 1 & 1 & 1 & 1 \\ 1 & 1 & 1 & 1 & 1 \\ 1 & 1 & 1 & 1 & 1 \\ 1 & 1 & 1 & 1 & 1 \end{bmatrix}$$

The operation works like this: keep this kernel above a pixel, add all the 25 pixels below this kernel,
take the average, and replace the central pixel with the new average value. This operation is continued
for all the pixels in the image. Try this code and check the result:

In [ ]:
import cv2 as cv
import ipywidgets as widgets
import numpy as np
from ipycanvas import Canvas, hold_canvas

img = cv.imread("../../../assets/opencv-logo-white.png")
img = cv.resize(img, (300, 300))
h, w = img.shape[:2]

# ---------- interactive widgets ----------
ksize_sl = widgets.IntSlider(value=5, min=1, max=31, step=2, description="Kernel size")

# ---------- canvases ----------
c_orig = Canvas(width=w, height=h)
c_orig.put_image_data(cv.cvtColor(img, cv.COLOR_BGR2RGB), 0, 0)
c_filt = Canvas(width=w, height=h)


def update(*_):
    k = ksize_sl.value
    kernel = np.ones((k, k), np.float32) / (k * k)
    dst = cv.filter2D(img, -1, kernel)
    with hold_canvas():
        c_filt.put_image_data(cv.cvtColor(dst, cv.COLOR_BGR2RGB), 0, 0)


ksize_sl.observe(update, "value")
update()

display(
    widgets.VBox(
        [
            ksize_sl,
            widgets.HBox(
                [c_orig, c_filt], layout=widgets.Layout(justify_content="space-around")
            ),
        ]
    )
)

Image Blurring (Image Smoothing)
--------------------------------

Image blurring is achieved by convolving the image with a low-pass filter kernel. It is useful for
removing noise. It actually removes high frequency content (eg: noise, edges) from the image. So
edges are blurred a little bit in this operation (there are also blurring techniques which don't
blur the edges). OpenCV provides four main types of blurring techniques.

### 1. Averaging

This is done by convolving an image with a normalized box filter. It simply takes the average of all
the pixels under the kernel area and replaces the central element. This is done by the function
**cv.blur()** or **cv.boxFilter()**. Check the docs for more details about the kernel. We should
specify the width and height of the kernel. A 3x3 normalized box filter would look like the below:

$$K =  \frac{1}{9} \begin{bmatrix} 1 & 1 & 1  \\ 1 & 1 & 1 \\ 1 & 1 & 1 \end{bmatrix}$$

> **Note:** If you don't want to use a normalized box filter, use **cv.boxFilter()**. Pass an argument
normalize=False to the function.

Check a sample demo below with a kernel of 5x5 size:

In [ ]:
import cv2 as cv
import ipywidgets as widgets
from ipycanvas import Canvas, hold_canvas

img = cv.imread("../../../assets/opencv-logo-white.png")
img = cv.resize(img, (300, 300))

h, w = img.shape[:2]

# ---------- interactive widgets ----------
ksize_sl = widgets.IntSlider(value=5, min=1, max=31, step=2, description="Kernel size")

# ---------- canvases ----------
c_orig = Canvas(width=w, height=h)
c_orig.put_image_data(cv.cvtColor(img, cv.COLOR_BGR2RGB), 0, 0)
c_blur = Canvas(width=w, height=h)


def update(*_):
    k = ksize_sl.value
    blur = cv.blur(img, (k, k))
    with hold_canvas():
        c_blur.put_image_data(cv.cvtColor(blur, cv.COLOR_BGR2RGB), 0, 0)


ksize_sl.observe(update, "value")
update()

display(
    widgets.VBox(
        [
            ksize_sl,
            widgets.HBox(
                [c_orig, c_blur], layout=widgets.Layout(justify_content="space-around")
            ),
        ]
    )
)

### 2. Gaussian Blurring

In this method, instead of a box filter, a Gaussian kernel is used. It is done with the function,
**cv.GaussianBlur()**. We should specify the width and height of the kernel which should be positive
and odd. We also should specify the standard deviation in the X and Y directions, sigmaX and sigmaY
respectively. If only sigmaX is specified, sigmaY is taken as the same as sigmaX. If both are given as
zeros, they are calculated from the kernel size. Gaussian blurring is highly effective in removing
Gaussian noise from an image.

If you want, you can create a Gaussian kernel with the function, **cv.getGaussianKernel()**.

The above code can be modified for Gaussian blurring:

In [ ]:
import cv2 as cv
import ipywidgets as widgets
from ipycanvas import Canvas, hold_canvas

img = cv.imread("../../../assets/opencv-logo-white.png")
img = cv.resize(img, (300, 300))
h, w = img.shape[:2]

# ---------- interactive widgets ----------
ksize_sl = widgets.IntSlider(value=5, min=1, max=31, step=2, description="Kernel size")
sigma_sl = widgets.FloatSlider(value=0, min=0, max=20, step=0.5, description="Sigma")

# ---------- canvases ----------
c_orig = Canvas(width=w, height=h)
c_orig.put_image_data(cv.cvtColor(img, cv.COLOR_BGR2RGB), 0, 0)
c_gauss = Canvas(width=w, height=h)


def update(*_):
    k = ksize_sl.value
    s = sigma_sl.value
    blur = cv.GaussianBlur(img, (k, k), s)
    with hold_canvas():
        c_gauss.put_image_data(cv.cvtColor(blur, cv.COLOR_BGR2RGB), 0, 0)


for wgt in (ksize_sl, sigma_sl):
    wgt.observe(update, "value")
update()

display(
    widgets.VBox(
        [
            widgets.HBox([ksize_sl, sigma_sl]),
            widgets.HBox(
                [c_orig, c_gauss], layout=widgets.Layout(justify_content="space-around")
            ),
        ]
    )
)

### 3. Median Blurring

Here, the function **cv.medianBlur()** takes the median of all the pixels under the kernel area and the central
element is replaced with this median value. This is highly effective against salt-and-pepper noise
in an image. Interestingly, in the above filters, the central element is a newly
calculated value which may be a pixel value in the image or a new value. But in median blurring,
the central element is always replaced by some pixel value in the image. It reduces the noise
effectively. Its kernel size should be a positive odd integer.

In this demo, I added a 50% noise to our original image and applied median blurring. Check the result:

In [ ]:
import cv2 as cv
import ipywidgets as widgets
from ipycanvas import Canvas, hold_canvas

img = cv.imread("../../../assets/opencv-logo-white.png")
img = cv.resize(img, (300, 300))
h, w = img.shape[:2]

# ---------- interactive widgets ----------
ksize_sl = widgets.IntSlider(value=5, min=1, max=31, step=2, description="Kernel size")
noise_sl = widgets.FloatSlider(value=0.5, min=0, max=1, step=0.05, description="Noise")

# ---------- canvases ----------
c_noisy = Canvas(width=w, height=h)
c_median = Canvas(width=w, height=h)


def add_salt_pepper(src, amount):
    out = src.copy()
    n = int(amount * src.size / 2)
    # salt (white)
    ys, xs = np.random.randint(0, src.shape[0], n), np.random.randint(
        0, src.shape[1], n
    )
    out[ys, xs] = 255
    # pepper (black)
    ys, xs = np.random.randint(0, src.shape[0], n), np.random.randint(
        0, src.shape[1], n
    )
    out[ys, xs] = 0
    return out


def update(*_):
    k = ksize_sl.value
    noisy = add_salt_pepper(img, noise_sl.value)
    median = cv.medianBlur(noisy, k)
    with hold_canvas():
        c_noisy.put_image_data(cv.cvtColor(noisy, cv.COLOR_BGR2RGB), 0, 0)
        c_median.put_image_data(cv.cvtColor(median, cv.COLOR_BGR2RGB), 0, 0)


for wgt in (ksize_sl, noise_sl):
    wgt.observe(update, "value")
update()

display(
    widgets.VBox(
        [
            widgets.HBox([ksize_sl, noise_sl]),
            widgets.HBox(
                [c_noisy, c_median],
                layout=widgets.Layout(justify_content="space-around"),
            ),
        ]
    )
)

### 4. Bilateral Filtering

**cv.bilateralFilter()** is highly effective in noise removal while keeping edges sharp. But the
operation is slower compared to other filters. We already saw that a Gaussian filter takes the
neighbourhood around the pixel and finds its Gaussian weighted average. This Gaussian filter is a
function of space alone, that is, nearby pixels are considered while filtering. It doesn't consider
whether pixels have almost the same intensity. It doesn't consider whether a pixel is an edge pixel or
not. So it blurs the edges also, which we don't want to do.

Bilateral filtering also takes a Gaussian filter in space, but one more Gaussian filter which is a
function of pixel difference. The Gaussian function of space makes sure that only nearby pixels are considered
for blurring, while the Gaussian function of intensity difference makes sure that only those pixels with
similar intensities to the central pixel are considered for blurring. So it preserves the edges since
pixels at edges will have large intensity variation.

The below sample shows use of a bilateral filter (For details on arguments, visit docs).

In [ ]:
import cv2 as cv
import ipywidgets as widgets
from ipycanvas import Canvas, hold_canvas

img = cv.imread("../../../assets/opencv-logo-white.png")
img = cv.resize(img, (300, 300))
h, w = img.shape[:2]

# ---------- interactive widgets ----------
d_sl = widgets.IntSlider(value=9, min=1, max=31, step=2, description="Diameter")
sc_sl = widgets.IntSlider(value=75, min=1, max=200, description="Sigma color")
ss_sl = widgets.IntSlider(value=75, min=1, max=200, description="Sigma space")

# ---------- canvases ----------
c_orig = Canvas(width=w, height=h)
c_orig.put_image_data(cv.cvtColor(img, cv.COLOR_BGR2RGB), 0, 0)
c_bilat = Canvas(width=w, height=h)


def update(*_):
    d = d_sl.value
    sc = sc_sl.value
    ss = ss_sl.value
    blur = cv.bilateralFilter(img, d, sc, ss)
    with hold_canvas():
        c_bilat.put_image_data(cv.cvtColor(blur, cv.COLOR_BGR2RGB), 0, 0)


for wgt in (d_sl, sc_sl, ss_sl):
    wgt.observe(update, "value")
update()

display(
    widgets.VBox(
        [
            widgets.HBox([d_sl, sc_sl, ss_sl]),
            widgets.HBox(
                [c_orig, c_bilat], layout=widgets.Layout(justify_content="space-around")
            ),
        ]
    )
)

Additional Resources
--------------------

1. Details about the [bilateral filtering](http://people.csail.mit.edu/sparis/bf_course/)